**Install Required Libraries**

In [3]:
pip install fake-useragent



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



**Import Required Libraries**

In [4]:
# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

import requests
import pandas as pd
import re
import time
import hashlib

from bs4 import BeautifulSoup
from tqdm import tqdm
from fake_useragent import UserAgent

from urllib.parse import urljoin

from reportlab.platypus import (
    SimpleDocTemplate,
    Table,
    TableStyle,
    Paragraph
)

from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet

**Define Governance URLs**

In [5]:
SEED_URLS = [

    "https://www.eacc.go.ke/",
    "https://www.iebc.or.ke/",
    "https://www.publicservice.go.ke/",
    "https://www.ecitizen.go.ke/",
    "https://immigration.go.ke/",
    "https://ntsa.go.ke/",
    "https://www.health.go.ke/",
    "https://www.parliament.go.ke/",
    "https://mygov.go.ke/",
    "https://www.kra.go.ke/",
    "https://www.odpp.go.ke/",
    "https://www.knchr.org/"

]

**Create a Web Request Function**

In [6]:
# ============================================================
# CELL 4 (Improved)
# ============================================================

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

ua = UserAgent()

HEADERS = {
    "User-Agent": ua.random
}

def fetch_page(url, timeout=30):

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=timeout,
            verify=False      # Ignore bad SSL certificates
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        time.sleep(1)

        return soup

    except Exception as e:

        print(f"\nFailed: {url}")
        print(e)

        return None

**DISCOVER ALL LINKS**

In [7]:
# ============================================================
# CELL 5: DISCOVER ALL LINKS
# ============================================================

all_links = []

for website in tqdm(SEED_URLS):

    soup = fetch_page(website)

    if soup is None:
        continue

    for link in soup.find_all("a", href=True):

        href = link["href"]

        # Convert relative URLs into absolute URLs
        full_url = urljoin(website, href)

        all_links.append(full_url)

print(f"Total links discovered: {len(all_links)}")

  0%|          | 0/12 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:57<00:00,  4.78s/it]

Total links discovered: 1536


**FILTER GOVERNANCE LINKS**

In [8]:
# ============================================================
# CELL 6: FILTER GOVERNANCE LINKS
# ============================================================

ARTICLE_KEYWORDS = [

    "news",
    "media",
    "press",
    "press-release",
    "announcement",
    "announcements",
    "notice",
    "notices",
    "alert",
    "alerts",
    "update",
    "updates",
    "article",
    "communication",
    "public",
    "events",
    "latest",
    "speech",
    "citizen",
    "media-centre",
    "media-center",
    "newsroom"

]

candidate_links = []

for link in all_links:

    lower = link.lower()

    if any(keyword in lower for keyword in ARTICLE_KEYWORDS):

        candidate_links.append(link)

print(f"Candidate governance pages: {len(candidate_links)}")

Candidate governance pages: 409


**REMOVE DUPLICATES**

In [9]:
# ============================================================
# CELL 7: REMOVE DUPLICATES
# ============================================================

candidate_links = list(set(candidate_links))

candidate_links = [

    url for url in candidate_links

    if url.startswith("http")

]

candidate_links.sort()

print(f"Unique links: {len(candidate_links)}")

Unique links: 170


**DOWNLOAD ARTICLES**

In [10]:
# ============================================================
# CELL 8: DOWNLOAD ARTICLES
# ============================================================

raw_articles = []

for url in tqdm(candidate_links):

    soup = fetch_page(url)

    if soup is None:
        continue

    title = ""

    if soup.title:
        title = soup.title.get_text(strip=True)

    paragraphs = soup.find_all("p")

    text = " ".join(

        p.get_text(" ", strip=True)

        for p in paragraphs

    )

    if len(text) < 200:
        continue

    raw_articles.append({

        "url": url,

        "title": title,

        "text": text

    })

print(f"Articles collected: {len(raw_articles)}")

 22%|██▏       | 37/170 [03:19<10:19,  4.66s/it]


Failed: https://www.eacc.go.ke/news-updates
404 Client Error: Not Found for url: https://www.eacc.go.ke/news-updates


 24%|██▎       | 40/170 [03:58<26:53, 12.41s/it]


Failed: https://www.ecitizen.go.ke/en/
HTTPSConnectionPool(host='www.ecitizen.go.ke', port=443): Read timed out. (read timeout=30)


 26%|██▋       | 45/170 [04:57<28:15, 13.56s/it]

...


Failed: https://www.knchr.org/Publications/Thematic-Reports/Group-Rights/Rights-of-Migrants
HTTPSConnectionPool(host='www.knchr.org', port=443): Max retries exceeded with url: /Publications/Thematic-Reports/Group-Rights/Rights-of-Migrants (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94C5FF0D0>: Failed to resolve 'www.knchr.org' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.knchr.org/Publications/Thematic-Reports/Group-Rights/Rights-of-Older-Persons
HTTPSConnectionPool(host='www.knchr.org', port=443): Max retries exceeded with url: /Publications/Thematic-Reports/Group-Rights/Rights-of-Older-Persons (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94C63D710>: Failed to resolve 'www.knchr.org' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.knchr.org/Publications/Thematic-Reports/Group-Rights/Rights-of-Persons-with-Disability-PWD
HTTPSConnectionPool(host='www.knchr.org', port=443): Max

 92%|█████████▏| 157/170 [1:39:12<01:14,  5.70s/it]


Failed: https://www.publicservice.go.ke/
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94D51E690>: Failed to resolve 'www.publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.publicservice.go.ke/#ekit_modal-popup-2d11ac0
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94D51D390>: Failed to resolve 'www.publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.publicservice.go.ke/#ekit_modal-popup-a97a2a8
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94D0DE510>: Failed to resolve 'www.publicservice.go.ke' ([Errno 11001] getaddrinfo fail

100%|██████████| 170/170 [1:39:12<00:00, 35.02s/it]


Failed: https://www.publicservice.go.ke/report-corruption/
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: /report-corruption/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94C602F50>: Failed to resolve 'www.publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.publicservice.go.ke/reports/
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: /reports/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94D51D310>: Failed to resolve 'www.publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.publicservice.go.ke/speeches/
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: /speeches/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94D51CB50>: Failed to resolve 'www.publicservice.go.ke' ([Errno 110

**SCRAPE EVERY DISCOVERED PAGE**

In [11]:
# ============================================================
# CELL 9: SCRAPE EVERY DISCOVERED PAGE
# ============================================================

raw_articles = []

for url in tqdm(candidate_links):

    soup = fetch_page(url)

    if soup is None:
        continue

    # ---------------------------
    # Get Title
    # ---------------------------

    if soup.title:
        title = soup.title.get_text(strip=True)
    else:
        title = ""

    # ---------------------------
    # Get Meta Description
    # ---------------------------

    description = ""

    meta = soup.find("meta", attrs={"name":"description"})

    if meta:
        description = meta.get("content","")

    # ---------------------------
    # Extract Paragraphs
    # ---------------------------

    paragraphs = soup.find_all("p")

    text = " ".join(

        p.get_text(" ", strip=True)

        for p in paragraphs

    )

    if len(text) < 150:

        continue

    raw_articles.append({

        "url": url,

        "title": title,

        "description": description,

        "text": text

    })

print()

print("Articles collected:", len(raw_articles))

 16%|█▌        | 27/170 [00:00<00:01, 132.50it/s]


Failed: http://publicservice.go.ke/about-us/
HTTPConnectionPool(host='publicservice.go.ke', port=80): Max retries exceeded with url: /about-us/ (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001F94C603550>: Failed to resolve 'publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: http://publicservice.go.ke/jobs/
HTTPConnectionPool(host='publicservice.go.ke', port=80): Max retries exceeded with url: /jobs/ (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001F94D0DF490>: Failed to resolve 'publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: http://publicservice.go.ke/news-updates/
HTTPConnectionPool(host='publicservice.go.ke', port=80): Max retries exceeded with url: /news-updates/ (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001F94E06CD10>: Failed to resolve 'publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: http://publicservice.go.ke/p

 24%|██▍       | 41/170 [00:00<00:01, 118.09it/s]


Failed: https://parliament.go.ke/index.php/invitation-submit-memoranda-notification-public-hearings-matter-consideration-national-assembly
HTTPSConnectionPool(host='parliament.go.ke', port=443): Max retries exceeded with url: /index.php/invitation-submit-memoranda-notification-public-hearings-matter-consideration-national-assembly (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94D0DF7D0>: Failed to resolve 'parliament.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://parliament.go.ke/invitation-submit-memoranda-notification-public-hearings-matter-consideration-national-assembly
HTTPSConnectionPool(host='parliament.go.ke', port=443): Max retries exceeded with url: /invitation-submit-memoranda-notification-public-hearings-matter-consideration-national-assembly (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94B27C750>: Failed to resolve 'parliament.go.ke' ([Errno 11001] getaddrinfo failed)"))

Fai

 38%|███▊      | 65/170 [00:00<00:00, 116.36it/s]

...


Failed: https://www.knchr.org/Publications/Thematic-Reports/Civil-and-Political-Rights/Prison-Reforms
HTTPSConnectionPool(host='www.knchr.org', port=443): Max retries exceeded with url: /Publications/Thematic-Reports/Civil-and-Political-Rights/Prison-Reforms (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94E06FBD0>: Failed to resolve 'www.knchr.org' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.knchr.org/Publications/Thematic-Reports/Civil-and-Political-Rights/Security-Sector
HTTPSConnectionPool(host='www.knchr.org', port=443): Max retries exceeded with url: /Publications/Thematic-Reports/Civil-and-Political-Rights/Security-Sector (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94C6013D0>: Failed to resolve 'www.knchr.org' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.knchr.org/Publications/Thematic-Reports/Civil-and-Political-Rights/Torture-Extrajudicial-Executions-Enforced-Disapp

 86%|████████▌ | 146/170 [00:01<00:00, 114.19it/s]


Failed: https://www.parliament.go.ke/index.php/the-national-assembly/committees/12/public-debt-privatization
HTTPSConnectionPool(host='www.parliament.go.ke', port=443): Max retries exceeded with url: /index.php/the-national-assembly/committees/12/public-debt-privatization (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94D0DF350>: Failed to resolve 'www.parliament.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.parliament.go.ke/index.php/the-national-assembly/committees/12/public-investments-committee-on-governance-education
HTTPSConnectionPool(host='www.parliament.go.ke', port=443): Max retries exceeded with url: /index.php/the-national-assembly/committees/12/public-investments-committee-on-governance-education (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94B9A8E50>: Failed to resolve 'www.parliament.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.parliament.go.ke/inde

100%|██████████| 170/170 [00:01<00:00, 121.61it/s]


Failed: https://www.publicservice.go.ke/newsletters/
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: /newsletters/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94B9A8B90>: Failed to resolve 'www.publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.publicservice.go.ke/partner-with-us
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: /partner-with-us (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94B9AA6D0>: Failed to resolve 'www.publicservice.go.ke' ([Errno 11001] getaddrinfo failed)"))

Failed: https://www.publicservice.go.ke/photo-gallery/
HTTPSConnectionPool(host='www.publicservice.go.ke', port=443): Max retries exceeded with url: /photo-gallery/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001F94C602310>: Failed to resolve 'www.publicservice.go.ke'

**Create a Dataframe**

In [12]:
# ============================================================
# CELL 10: CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(raw_articles)

print(df.shape)

df.head()

(0, 0)


""


**REMOVE DUPLICATES**

In [13]:
# ============================================================
# CELL 11: REMOVE DUPLICATES
# ============================================================

df.drop_duplicates(
    subset=["url"],
    inplace=True
)

df.drop_duplicates(
    subset=["text"],
    inplace=True
)

df.reset_index(drop=True, inplace=True)

print("Remaining articles:", len(df))

Remaining articles: 0


**Clean Article Text**

In [14]:
# ============================================================
# CELL 12: CLEAN ARTICLE TEXT
# ============================================================

def clean_text(text):

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

df["text"] = df["text"].apply(clean_text)

df["title"] = df["title"].apply(clean_text)

df["description"] = df["description"].fillna("").apply(clean_text)

print(df.head())

KeyError: 'text'

**INSTALL TRAFILATURA**

In [ ]:
# ============================================================
# CELL 13: INSTALL TRAFILATURA
# ============================================================

%pip install trafilatura


  Using cached trafilatura-2.1.0-py3-none-any.whl.metadata (13 kB)
  Using cached courlan-1.4.0-py3-none-any.whl.metadata (18 kB)
  Using cached htmldate-1.10.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached justext-3.0.2-py2.py3-none-any.whl.metadata (7.3 kB)
  Using cached babel-2.18.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached tld-0.13.2-py2.py3-none-any.whl.metadata (11 kB)
  Using cached dateparser-1.4.1-py3-none-any.whl.metadata (22 kB)
  Using cached tzlocal-5.4.4-py3-none-any.whl.metadata (7.7 kB)
  Using cached lxml_html_clean-0.4.5-py3-none-any.whl.metadata (2.4 kB)
Using cached trafilatura-2.1.0-py3-none-any.whl (134 kB)
Using cached courlan-1.4.0-py3-none-any.whl (34 kB)
Using cached babel-2.18.0-py3-none-any.whl (10.2 MB)
Using cached htmldate-1.10.0-py3-none-any.whl (31 kB)
Using cached dateparser-1.4.1-py3-none-any.whl (300 kB)
Using cached justext-3.0.2-py2.py3-none-any.whl (837 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- --

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Import Trafilatura**

In [ ]:
# ============================================================
# CELL 14: IMPORT TRAFILATURA
# ============================================================

import trafilatura

**ARTICLE EXTRACTION FUNCTION**

In [ ]:
# ============================================================
# CELL 15: ARTICLE EXTRACTION FUNCTION
# ============================================================

def extract_article(url):
    """
    Download a webpage and extract the main article using Trafilatura.
    Returns:
        Clean article text or None.
    """

    try:

        downloaded = trafilatura.fetch_url(url)

        if downloaded is None:
            return None

        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            include_images=False,
            favor_precision=True
        )

        return text

    except Exception as e:

        print(url)
        print(e)

        return None

**EXTRACT CLEAN ARTICLE TEXT**

In [ ]:
# ============================================================
# CELL 16: EXTRACT CLEAN ARTICLE TEXT
# ============================================================

clean_articles = []

for i, row in tqdm(df.iterrows(), total=len(df)):

    article = extract_article(row["url"])

    if article is None:
        continue

    if len(article) < 150:
        continue

    clean_articles.append({

        "url": row["url"],

        "title": row["title"],

        "description": row["description"],

        "text": article

    })

clean_df = pd.DataFrame(clean_articles)

print()

print("Clean Articles:", len(clean_df))

clean_df.head()

  0%|          | 0/107 [00:00<?, ?it/s]

100%|██████████| 107/107 [07:01<00:00,  3.94s/it]



Clean Articles: 79


,url,title,description,text
0,http://publicservice.go.ke/about-us/,About Us - Public Service Commission,,About Us\n1954\nYear Established\n20K +\nYouth...
1,http://publicservice.go.ke/news-updates/,News Updates - Public Service Commission,,The Public Service Commission (PSC) will roll ...
2,http://publicservice.go.ke/programmes/,Programmes - Public Service Commission,,Programmes\nPromote Inclusivity\nPromote Entre...
3,https://accounts.ecitizen.go.ke/en,Home · Government of Kenya services simplified,"Government of Kenya online portal offers 16,00...",Government of Kenya services\n simp...
4,https://dis.ecitizen.go.ke/,Welcome,Directorate of Immigration,The Directorate of Immigration Services is a D...


**SAVE CLEAN DATASET**

In [ ]:
# ============================================================
# CELL 17: SAVE CLEAN DATASET
# ============================================================

clean_df.to_excel(
    "01_Clean_Governance_Articles.xlsx",
    index=False
)

print("Backup saved successfully.")

Backup saved successfully.


**DEFINING PSA VOCABULARY**

In [ ]:
# ============================================================
# CELL 18: PSA VOCABULARY
# ============================================================

ACTION_WORDS = [

    "register",
    "verify",
    "report",
    "apply",
    "submit",
    "renew",
    "collect",
    "attend",
    "participate",
    "visit",
    "download",
    "call",
    "pay",
    "complete",
    "obtain",
    "check",
    "update",
    "book",
    "access"

]

URGENCY_WORDS = [

    "deadline",
    "urgent",
    "immediately",
    "before",
    "last day",
    "expires",
    "warning",
    "alert",
    "important",
    "reminder",
    "must",
    "required",
    "mandatory"

]

AUDIENCE_WORDS = [

    "citizens",
    "residents",
    "public",
    "voters",
    "applicants",
    "farmers",
    "students",
    "businesses",
    "employers",
    "taxpayers",
    "drivers",
    "parents",
    "workers"

]

GOVERNMENT_AGENCIES = [

    "IEBC",
    "EACC",
    "NTSA",
    "KRA",
    "NEMA",
    "SHA",
    "Ministry",
    "Government",
    "County",
    "eCitizen",
    "Parliament",
    "Senate",
    "Commission"

]

**PSA SCORING ENGINE**

In [ ]:
# ============================================================
# CELL 19: PSA SCORING ENGINE
# ============================================================

def psa_score(text):

    if text is None:
        return 0

    text = text.lower()

    score = 0

    # -----------------------
    # Action words
    # -----------------------

    score += sum(word in text for word in ACTION_WORDS)

    # -----------------------
    # Urgency
    # -----------------------

    score += 2 * sum(word in text for word in URGENCY_WORDS)

    # -----------------------
    # Target audience
    # -----------------------

    score += sum(word in text for word in AUDIENCE_WORDS)

    # -----------------------
    # Government agencies
    # -----------------------

    score += sum(word.lower() in text for word in GOVERNMENT_AGENCIES)

    return score

**COMPUTE PSA SCORE**

In [ ]:
# ============================================================
# CELL 20: COMPUTE PSA SCORE
# ============================================================

clean_df["PSA_Score"] = clean_df["text"].apply(psa_score)

clean_df.head()

,url,title,description,text,PSA_Score
0,http://publicservice.go.ke/about-us/,About Us - Public Service Commission,,About Us\n1954\nYear Established\n20K +\nYouth...,4
1,http://publicservice.go.ke/news-updates/,News Updates - Public Service Commission,,The Public Service Commission (PSC) will roll ...,4
2,http://publicservice.go.ke/programmes/,Programmes - Public Service Commission,,Programmes\nPromote Inclusivity\nPromote Entre...,4
3,https://accounts.ecitizen.go.ke/en,Home · Government of Kenya services simplified,"Government of Kenya online portal offers 16,00...",Government of Kenya services\n simp...,8
4,https://dis.ecitizen.go.ke/,Welcome,Directorate of Immigration,The Directorate of Immigration Services is a D...,5


**FILTER PSA ARTICLES**

In [ ]:
# ============================================================
# CELL 21: FILTER PSA ARTICLES
# ============================================================

PSA_THRESHOLD = 5

psa_df = clean_df[
    clean_df["PSA_Score"] >= PSA_THRESHOLD
].copy()

print(f"PSAs Detected: {len(psa_df)}")

psa_df.head()

PSAs Detected: 41


,url,title,description,text,PSA_Score
3,https://accounts.ecitizen.go.ke/en,Home · Government of Kenya services simplified,"Government of Kenya online portal offers 16,00...",Government of Kenya services\n simp...,8
4,https://dis.ecitizen.go.ke/,Welcome,Directorate of Immigration,The Directorate of Immigration Services is a D...,5
5,https://eacc.go.ke/en/default/feed/eacc-and-bu...,EACC urges counties to refrain from graft for ...,,EACC urges counties to refrain from graft for ...,15
6,https://eacc.go.ke/en/default/feed/two-public-...,Two public officers arrested for diversion of ...,,Two public officers arrested for diversion of ...,9
7,https://eacc.go.ke/en/default/news-updates/,News & Updates - EACC,,Ethics and Anti-Corruption Commission gathers ...,5


**SAVE PSA DATASET TO CSV**

In [ ]:
# ============================================================
# CELL 22: SAVE PSA DATASET TO CSV
# ============================================================

import os

# Create output folder if it doesn't exist
os.makedirs("output", exist_ok=True)

# Save the detected PSAs
csv_file = "output/Governance_PSA_Dataset.csv"

psa_df.to_csv(
    csv_file,
    index=False,
    encoding="utf-8-sig"   # Ensures Excel opens it correctly
)

print("="*60)
print("Dataset successfully saved!")
print(f"Location: {csv_file}")
print(f"Total PSA Records: {len(psa_df)}")
print("="*60)

PermissionError: [Errno 13] Permission denied: 'output/Governance_PSA_Dataset.csv'

**VERIFY**

In [ ]:
# ============================================================
# CELL 23: READ THE CSV BACK
# ============================================================

saved_df = pd.read_csv("output/Governance_PSA_Dataset.csv")

print(saved_df.shape)

saved_df.head()

(41, 5)


,url,title,description,text,PSA_Score
0,https://accounts.ecitizen.go.ke/en,Home · Government of Kenya services simplified,"Government of Kenya online portal offers 16,00...",Government of Kenya services\n simp...,8
1,https://dis.ecitizen.go.ke/,Welcome,Directorate of Immigration,The Directorate of Immigration Services is a D...,5
2,https://eacc.go.ke/en/default/feed/eacc-and-bu...,EACC urges counties to refrain from graft for ...,NaN,EACC urges counties to refrain from graft for ...,15
3,https://eacc.go.ke/en/default/feed/two-public-...,Two public officers arrested for diversion of ...,NaN,Two public officers arrested for diversion of ...,9
4,https://eacc.go.ke/en/default/news-updates/,News & Updates - EACC,NaN,Ethics and Anti-Corruption Commission gathers ...,5


# **Transformer**

In [ ]:
# ============================================================
# CELL 24
# ============================================================

!pip install -U transformers torch accelerate sentencepiece

In [ ]:
# ============================================================
# CELL 25
# ============================================================

import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

print("TensorFlow disabled.")

TensorFlow disabled.


In [ ]:
%pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.7.1+cu118
Uninstalling torch-2.7.1+cu118:
  Successfully uninstalled torch-2.7.1+cu118
Found existing installation: torchaudio 2.7.1+cu118
Uninstalling torchaudio-2.7.1+cu118:
  Successfully uninstalled torchaudio-2.7.1+cu118
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/121.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/121.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/121.9 MB ? eta -:--:--
   ---------------------------------------- 1.0/121.9 MB 2.8 MB/s eta 0:00:44
    --------------------------------------- 2.4/121.9 MB 4.3 MB/s eta 0:00:28
   -- ------------------------------------- 6.3/121.9 MB 8.6 MB/s eta 0:00:14
   ----- ---------------------------------- 16.8/121.9 MB 17.9 MB/s eta 0:00:06
   -------- ------------------------------- 26.7/121.9 MB 23.5 MB/s eta 0:00:05
   --------- ------------------------------ 28.0/121.9 MB 24.4 MB/s eta 0:00:04
   --------- ------------------------------ 28.3/121.9 MB 20.4 MB/s eta 0:00:05
   --------- ------------------------------ 29.6/121.9 MB 18.6 MB/s eta 0:00:05
   ------------ --------------------------- 37.2/121.9 MB 18.8 MB/s eta 0:00:05
   -------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys

print(sys.version)
print(sys.executable)

3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
c:\Users\Admin\AppData\Local\Programs\Python\Python311\python.exe


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# ============================================================
# CELL 26
# ============================================================

import torch

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

print(torch.__version__)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# ============================================================
# CELL 25: IMPORT NLP LIBRARIES
# ============================================================

from transformers import pipeline

print("Transformers imported successfully.")

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers imported successfully.


In [ ]:
# ============================================================
# CELL 27
# ============================================================

MODEL_NAME = "MoritzLaurer/deberta-v3-base-zeroshot-v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print("Running on:", device)

In [ ]:
# ============================================================
# CELL 26: LOAD THE MODEL
# ============================================================

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Model loaded successfully.")

: 

In [ ]:
# ============================================================
# CELL 27: GOVERNANCE LABELS
# ============================================================

labels = [

    "Anti-Corruption",

    "Public Participation",

    "Elections and Voter Education",

    "Public Service Delivery",

    "Devolution and Local Governance"

]

print(labels)

In [ ]:
# ============================================================
# CELL 28: TEST THE MODEL
# ============================================================

sample = """

The Independent Electoral and Boundaries Commission
reminds all registered voters to verify their voter
registration details before the registration deadline.

"""

result = classifier(
    sample,
    labels
)

print(result)

In [ ]:
# ============================================================
# CELL 29: CLASSIFY A GOVERNANCE PSA
# ============================================================

def classify_psa(text):

    if not isinstance(text, str):
        return "Unknown", 0.0

    try:

        prediction = classifier(
            text[:1000],   # Limit input length
            labels
        )

        return (

            prediction["labels"][0],

            prediction["scores"][0]

        )

    except Exception:

        return "Unknown", 0.0

In [ ]:
!pip install tqdm


Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# ============================================================
# CELL 30: CLASSIFY ALL PSAs
# ============================================================
from tqdm import tqdm
predictions = []

confidences = []

for article in tqdm(psa_df["text"]):

    category, confidence = classify_psa(article)

    predictions.append(category)

    confidences.append(confidence)

psa_df["Category"] = predictions

psa_df["Confidence"] = confidences

psa_df.head()

NameError: name 'psa_df' is not defined